# Notebook 04: Evaluasi RAG + Context Reranking — PubMedQA

Konfigurasi **RAG + Context Reranking (CR)**.
Retriever: BM25 (top-20 kandidat) → CrossEncoder rerank → top-5.

**Pipeline:**
1. BM25 retrieve top-20 kandidat (lebih banyak dari baseline)
2. CrossEncoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) score tiap pasangan (query, doc)
3. Ambil top-5 berdasarkan skor reranker
4. Generate jawaban dengan LLM (prompt identik dengan baseline)

**Hipotesis:** BM25 mengambil dokumen yang lexically similar tapi tidak selalu semantically relevant.
CrossEncoder yang membaca pasangan (query, doc) secara bersamaan diharapkan menghasilkan
konteks yang lebih relevan → faithfulness dan context recall lebih tinggi.

**Perbandingan:** Baseline (BM25 top-5) vs QR (BM25+QR top-5) vs **CR (BM25 top-20 → Rerank top-5)**

## 1. Impor Library

In [1]:
import os, sys, json, pickle, time, re, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime
from collections import Counter

from rank_bm25 import BM25Okapi
import ollama
from datasets import load_dataset

# CrossEncoder untuk reranking
try:
    from sentence_transformers import CrossEncoder
    print('sentence-transformers tersedia.')
except ImportError:
    print('sentence-transformers belum terinstall!')
    print('Jalankan: pip install sentence-transformers')
    raise

warnings.filterwarnings('ignore', category=DeprecationWarning)
from ragas import EvaluationDataset, SingleTurnSample, evaluate, RunConfig
from ragas.metrics import (
    _Faithfulness,
    _ResponseRelevancy,
    _LLMContextPrecisionWithReference,
    _LLMContextRecall
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaLLM, OllamaEmbeddings

print('Semua library berhasil diimpor!')
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__} | Pandas: {pd.__version__}')

C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence-transformers tersedia.
Semua library berhasil diimpor!
Python: 3.11.9 | NumPy: 2.3.5 | Pandas: 2.3.3


## 2. Konfigurasi

In [2]:
LLM_MODEL        = 'llama3.2'
EMBED_MODEL      = 'nomic-embed-text'       # Hanya untuk RAGAS
RERANKER_MODEL   = 'cross-encoder/ms-marco-MiniLM-L-6-v2'  # Lightweight cross-encoder

TOP_K_CANDIDATES = 20   # BM25 ambil 20 kandidat
TOP_K_RETRIEVAL  = 5    # Setelah rerank, ambil top-5 (sama dengan baseline)

DATASET_NAME   = 'qiaojin/PubMedQA'
DATASET_SUBSET = 'pqa_labeled'
MAX_SAMPLES    = 500

TEMPERATURE = 0.0
SEED        = 42

NOTEBOOK_DIR    = Path('.')
BM25_INDEX_PATH = NOTEBOOK_DIR / 'pubmedqa_bm25.pkl'  # Shared dengan baseline & QR
RESULTS_DIR     = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

CONFIG_NAME    = 'cr'
PHASE1_PATH    = RESULTS_DIR / f'{CONFIG_NAME}_phase1_answers.json'
PHASE2_PATH    = RESULTS_DIR / f'{CONFIG_NAME}_phase2_ragas.json'
FINAL_CSV_PATH = RESULTS_DIR / f'{CONFIG_NAME}_results.csv'

BASELINE_PHASE1_PATH = RESULTS_DIR / 'baseline_phase1_answers.json'
BASELINE_PHASE2_CUSTOM_PATH = RESULTS_DIR / 'baseline_phase2_custom.json'
QR_PHASE1_PATH       = RESULTS_DIR / 'qr_phase1_answers.json'
QR_PHASE2_CUSTOM_PATH = RESULTS_DIR / 'qr_phase2_custom.json'

print('Konfigurasi:')
print(f'  LLM          : {LLM_MODEL}')
print(f'  Reranker     : {RERANKER_MODEL}')
print(f'  Retriever    : BM25 top-{TOP_K_CANDIDATES} -> CrossEncoder -> top-{TOP_K_RETRIEVAL}')
print(f'  Sampel       : {MAX_SAMPLES}')
print(f'  Config       : {CONFIG_NAME}')

Konfigurasi:
  LLM          : llama3.2
  Reranker     : cross-encoder/ms-marco-MiniLM-L-6-v2
  Retriever    : BM25 top-20 -> CrossEncoder -> top-5
  Sampel       : 500
  Config       : cr


## 3. Data Classes dan Tokenizer BM25

In [3]:
@dataclass
class Document:
    text         : str
    pubid        : str
    question     : str
    section_label: str
    answer       : str
    decision     : str

@dataclass
class RetrievalResult:
    document      : Document
    score         : float           # BM25 score (kandidat awal)
    reranker_score: float = 0.0     # CrossEncoder score (setelah rerank)


def tokenize_bm25(text: str) -> List[str]:
    """Tokenizer untuk BM25: hapus tanda baca, lowercase, split spasi."""
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower()).split()


sample_text = 'Does aspirin (75mg) reduce myocardial infarction risk?'
print(f'Tokenisasi BM25: {tokenize_bm25(sample_text)}')
print('Data classes dan tokenizer siap.')

Tokenisasi BM25: ['does', 'aspirin', '75mg', 'reduce', 'myocardial', 'infarction', 'risk']
Data classes dan tokenizer siap.


## 4. Muat Dataset dan Bangun BM25 Index

BM25 index di-share dengan notebook baseline dan QR — gunakan file `pubmedqa_bm25.pkl` yang sama.
Jika sudah ada, proses muat hanya butuh beberapa detik.

In [4]:
def load_pubmedqa(subset=DATASET_SUBSET, max_samples=MAX_SAMPLES):
    print(f'Memuat PubMedQA ({subset})...')
    dataset = load_dataset(DATASET_NAME, subset, trust_remote_code=True)
    data    = dataset['train']
    if max_samples and len(data) > max_samples:
        data = data.select(range(max_samples))
    print(f'Dimuat {len(data)} sampel')
    return data


def prepare_documents(data) -> List[Document]:
    docs = []
    for item in data:
        pubid = str(item['pubid'])
        for ctx, label in zip(item['context']['contexts'], item['context']['labels']):
            docs.append(Document(
                text=ctx.strip(), pubid=pubid,
                question=item['question'], section_label=label,
                answer=item['long_answer'], decision=item['final_decision']
            ))
    print(f'Total potongan dokumen: {len(docs)}')
    return docs


def load_or_build_bm25(data) -> Tuple[BM25Okapi, List[Document]]:
    if BM25_INDEX_PATH.exists():
        print(f'Memuat BM25 index dari {BM25_INDEX_PATH}...')
        with open(BM25_INDEX_PATH, 'rb') as f:
            saved = pickle.load(f)
        print(f'Dimuat: {len(saved["documents"])} dokumen')
        return saved['bm25'], saved['documents']
    else:
        print('Membangun BM25 index...')
        documents = prepare_documents(data)
        tokenized = [tokenize_bm25(d.text) for d in documents]
        bm25      = BM25Okapi(tokenized)
        with open(BM25_INDEX_PATH, 'wb') as f:
            pickle.dump({'bm25': bm25, 'documents': documents}, f)
        print(f'Index disimpan ke {BM25_INDEX_PATH}')
        return bm25, documents


t0 = time.time()
pubmedqa_data         = load_pubmedqa()
bm25_index, documents = load_or_build_bm25(pubmedqa_data)
print(f'Selesai dalam {time.time()-t0:.1f} detik')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Memuat PubMedQA (pqa_labeled)...


Dimuat 500 sampel
Memuat BM25 index dari pubmedqa_bm25.pkl...
Dimuat: 1706 dokumen
Selesai dalam 4.9 detik


## 5. Muat CrossEncoder Reranker

**Model:** `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Dilatih pada MS MARCO passage ranking (passage retrieval untuk QA)
- Lightweight: 6-layer MiniLM, ~22M parameter
- Input: pasangan (query, passage) → output: relevance score
- Tidak memerlukan GPU — berjalan di CPU

**Kenapa tidak bi-encoder?**
Bi-encoder encode query dan doc secara terpisah (embedding cosine similarity).
Cross-encoder membaca keduanya sekaligus → representasi interaksi lebih kaya → lebih akurat tapi lebih lambat.
Untuk reranking top-20 (bukan full corpus), cross-encoder trade-off latency-nya ok.

In [5]:
print(f'Memuat CrossEncoder: {RERANKER_MODEL}...')
print('(Download ~85MB sekali, lalu di-cache)')
t0 = time.time()
cross_encoder = CrossEncoder(RERANKER_MODEL)
print(f'CrossEncoder siap dalam {time.time()-t0:.1f} detik')

# Smoke test
_pairs = [
    ('Does aspirin prevent heart attacks?', 'Aspirin reduces platelet aggregation and is used in cardiovascular prevention.'),
    ('Does aspirin prevent heart attacks?', 'Weather patterns affect agricultural yields in tropical regions.'),
]
_scores = cross_encoder.predict(_pairs)
print(f'\nSmoke test CrossEncoder:')
print(f'  Relevan   : {_scores[0]:.4f}')
print(f'  Tidak relevan: {_scores[1]:.4f}')
assert _scores[0] > _scores[1], 'CrossEncoder gagal membedakan relevan vs tidak!'
print('Reranker berfungsi dengan benar.')

Memuat CrossEncoder: cross-encoder/ms-marco-MiniLM-L-6-v2...
(Download ~85MB sekali, lalu di-cache)


C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ricky Wijaya\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|███████████████████████| 105/105 [00:00<0

CrossEncoder siap dalam 21.1 detik

Smoke test CrossEncoder:
  Relevan   : 4.9414
  Tidak relevan: -11.1675
Reranker berfungsi dengan benar.


## 6. Fungsi Retrieval dengan Context Reranking (BM25 → CrossEncoder)

In [6]:
def retrieve_with_reranking(
    query: str,
    k_candidates: int = TOP_K_CANDIDATES,
    k_final: int = TOP_K_RETRIEVAL
) -> List[RetrievalResult]:
    """
    Retrieval dua tahap:
      1. BM25: ambil top-k_candidates (20) dokumen kandidat
      2. CrossEncoder: rerank kandidat, ambil top-k_final (5)

    Returns list RetrievalResult dengan atribut:
      .score          = BM25 score asli
      .reranker_score = CrossEncoder score setelah reranking
    """
    # Tahap 1: BM25 retrieval
    tokens    = tokenize_bm25(query)
    scores    = bm25_index.get_scores(tokens)
    top_cands = np.argsort(scores)[::-1][:k_candidates]
    candidates = [
        RetrievalResult(document=documents[i], score=float(scores[i]))
        for i in top_cands
    ]

    # Tahap 2: CrossEncoder reranking
    pairs = [(query, r.document.text) for r in candidates]
    reranker_scores = cross_encoder.predict(pairs)

    for r, rs in zip(candidates, reranker_scores):
        r.reranker_score = float(rs)

    # Sort by reranker score, ambil top-k_final
    reranked = sorted(candidates, key=lambda r: r.reranker_score, reverse=True)
    return reranked[:k_final]


# Test retrieval
test_q = 'Does aspirin reduce the risk of myocardial infarction?'
test_r = retrieve_with_reranking(test_q)
print(f'Query: {test_q}')
print(f'\nTop-{TOP_K_RETRIEVAL} dokumen (BM25 top-{TOP_K_CANDIDATES} → CrossEncoder):')  
for i, r in enumerate(test_r, 1):
    print(f'  [{i}] BM25={r.score:.2f} | Reranker={r.reranker_score:.4f} | {r.document.section_label} | {r.document.text[:80]}...')

Query: Does aspirin reduce the risk of myocardial infarction?

Top-5 dokumen (BM25 top-20 → CrossEncoder):
  [1] BM25=12.18 | Reranker=-2.5403 | BACKGROUND AND PURPOSE | In primary and secondary prevention trials, statins have been shown to reduce th...
  [2] BM25=16.24 | Reranker=-3.1160 | OBJECTIVE | Myocardial damage that is associated with percutaneous coronary intervention (PC...
  [3] BM25=23.49 | Reranker=-5.1495 | DESIGN | Within a prospective, population-based cohort study individuals without history ...
  [4] BM25=15.69 | Reranker=-5.3794 | BACKGROUND | It has recently been shown that non-high density lipoprotein cholesterol (non-HD...
  [5] BM25=19.31 | Reranker=-5.6010 | METHODS | Of the 9681 women and 8888 men who attended risk assessment from 1967-1991, with...


## 7. Prompt Generasi dan Fungsi Generate

Prompt **identik** dengan baseline — isolasi variabel: hanya retrieval yang berubah, bukan generation.

In [7]:
GENERATION_PROMPT = (
    'You are a medical research assistant. '
    'Answer a biomedical yes/no/maybe question based solely on the provided scientific abstracts.\n\n'
    'Context from medical literature:\n{context}\n\n'
    'Question: {question}\n\n'
    'Instructions:\n'
    '- Carefully read the context and assess whether it supports or refutes the question.\n'
    '- Provide a brief explanation (2-3 sentences) using ONLY the information above.\n'
    '- End your response with EXACTLY ONE of these words on its own line: yes, no, or maybe.\n'
    '  - yes   : the evidence supports the hypothesis, even if not perfectly conclusive\n'
    '  - no    : the evidence refutes or does not support the hypothesis\n'
    '  - maybe : ONLY if the evidence is directly contradictory (some findings say yes,\n'
    '            others say no), or if the context contains no relevant information at all\n'
    '- IMPORTANT: If the evidence leans in one direction, even partially, choose yes or no.\n'
    '  Do NOT use maybe simply because the evidence is limited or not 100% certain.\n\n'
    'Answer:'
)


def generate_answer(query: str, retrieved: List[RetrievalResult]) -> str:
    """Generate jawaban. Context Reranking: konteks sudah di-rerank oleh CrossEncoder."""
    context = '\n\n'.join(
        f'[{i}] ({r.document.section_label}): {r.document.text}'
        for i, r in enumerate(retrieved, 1)
    )
    response = ollama.generate(
        model=LLM_MODEL,
        prompt=GENERATION_PROMPT.format(context=context, question=query),
        options={'temperature': TEMPERATURE, 'seed': SEED, 'num_predict': 300}
    )
    return response['response'].strip()


# Test
test_ans = generate_answer(test_q, test_r)
print('Output generation:')
print('-' * 60)
print(test_ans)
print('-' * 60)

Output generation:
------------------------------------------------------------
Based on the provided scientific abstracts, there is no direct evidence to support or refute the question of whether aspirin reduces the risk of myocardial infarction. However, statins have been shown to reduce the risk of stroke and have antiatherothrombotic properties, which may indirectly suggest a potential benefit of aspirin in reducing cardiovascular risk.

The context does not provide any information about aspirin specifically, but rather focuses on statins and other lipid-lowering therapies. Therefore, based on the available information, it is difficult to make a definitive statement about the effect of aspirin on myocardial infarction risk.

maybe
------------------------------------------------------------


## 8. Ekstraksi Label yes/no/maybe

In [8]:
def extract_label(answer: str) -> str:
    """
    Ekstrak prediksi yes/no/maybe dari teks jawaban.
    Strategi (berurutan hingga ditemukan):
      1. Kata standalone di 3 baris terakhir (non-kosong)
      2. Kata standalone di seluruh teks
      3. Default ke 'maybe'
    """
    lines = [l.strip().lower() for l in answer.split('\n') if l.strip()]
    for line in reversed(lines[-3:]):
        word = re.sub(r'[^a-z]', '', line)
        if word in ('yes', 'no', 'maybe'):
            return word
    for label in ('yes', 'no', 'maybe'):
        if re.search(r'\b' + label + r'\b', answer.lower()):
            return label
    return 'maybe'


cases = [
    ('Strong evidence.\nyes',    'yes'),
    ('No effect found.\nno',     'no'),
    ('Mixed results.\nmaybe',    'maybe'),
    ('Verdict: yes.',             'yes'),
    ('Totally unclear.',          'maybe'),
]
print('Unit test extract_label:')
all_ok = True
for txt, exp in cases:
    pred = extract_label(txt)
    ok   = pred == exp
    all_ok = all_ok and ok
    print(f'  [{"PASS" if ok else "FAIL"}] pred={pred!r} expected={exp!r}')
print(f'\nSemua lulus: {all_ok}')
print(f'Label dari test answer: {extract_label(test_ans)!r}')

Unit test extract_label:
  [PASS] pred='yes' expected='yes'
  [PASS] pred='no' expected='no'
  [PASS] pred='maybe' expected='maybe'
  [PASS] pred='yes' expected='yes'
  [PASS] pred='maybe' expected='maybe'

Semua lulus: True
Label dari test answer: 'maybe'


## 9. Setup RAGAS dengan Ollama

In [9]:
ragas_llm = LangchainLLMWrapper(OllamaLLM(model=LLM_MODEL, temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model=EMBED_MODEL))

faithfulness_m  = _Faithfulness(llm=ragas_llm)
relevancy_m     = _ResponseRelevancy(llm=ragas_llm, embeddings=ragas_emb)
ctx_precision_m = _LLMContextPrecisionWithReference(llm=ragas_llm)
ctx_recall_m    = _LLMContextRecall(llm=ragas_llm)

RAGAS_RUN_CONFIG = RunConfig(max_workers=1, max_retries=2, timeout=120)

print('RAGAS setup selesai (max_workers=1 untuk local LLM).')

RAGAS setup selesai (max_workers=1 untuk local LLM).


---
## Setup Evaluator Custom (Zero-NaN)

Custom evaluator: faithfulness + context_recall tanpa NaN.
Identik dengan notebook baseline dan QR.

In [10]:
def _split_sentences(text: str) -> List[str]:
    """Pecah teks menjadi kalimat. Filter kalimat terlalu pendek (<15 char)."""
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in parts if len(s.strip()) >= 15]


def _llm_yes_no(prompt: str) -> bool:
    """Tanya LLM ya/tidak. Return True=yes, False=no. Fallback ke False jika gagal."""
    try:
        resp = ollama.generate(
            model=LLM_MODEL, prompt=prompt,
            options={'temperature': 0, 'seed': SEED, 'num_predict': 10}
        )
        return 'yes' in resp['response'].strip().lower()[:15]
    except Exception:
        return False


def compute_faithfulness(answer: str, contexts: List[str]) -> float:
    sentences = _split_sentences(answer)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement directly supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    supported = sum(
        1 for s in sentences
        if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s))
    )
    return supported / len(sentences)


def compute_context_recall(reference: str, contexts: List[str]) -> float:
    sentences = _split_sentences(reference)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    covered = sum(
        1 for s in sentences
        if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s))
    )
    return covered / len(sentences)


def evaluate_custom(question: str, answer: str,
                    contexts: List[str], reference: str) -> Dict:
    return {
        'faithfulness'  : compute_faithfulness(answer, contexts),
        'context_recall': compute_context_recall(reference, contexts),
    }


# Smoke test
_test_ctx  = ['Aspirin reduces blood clotting and is used for heart attack prevention.']
_test_ans  = 'Aspirin helps prevent heart attacks. It works by reducing clotting.'
_test_ref  = 'Aspirin is used for heart attack prevention by reducing blood clotting.'
_r = evaluate_custom('Does aspirin prevent heart attacks?', _test_ans, _test_ctx, _test_ref)
print('Smoke test evaluate_custom:')
print(f'  faithfulness   = {_r["faithfulness"]:.3f}  (expected > 0)')
print(f'  context_recall = {_r["context_recall"]:.3f} (expected > 0)')
print('Zero-NaN evaluator siap.')

Smoke test evaluate_custom:
  faithfulness   = 1.000  (expected > 0)
  context_recall = 1.000 (expected > 0)
Zero-NaN evaluator siap.


---
## DEMO: Uji Coba 5 Sampel

Verifikasi pipeline CR berjalan dengan benar.
Tampilkan BM25 kandidat vs hasil reranking untuk tiap sampel.

> Estimasi: ~2–3 menit

In [11]:
DEMO_SIZE    = 5
demo_results = []

print(f'DEMO: {DEMO_SIZE} sampel pertama (RAG + Context Reranking)')
print('=' * 70)

for i in range(DEMO_SIZE):
    s          = pubmedqa_data[i]
    q, gt, ref = s['question'], s['final_decision'], s['long_answer']

    # Retrieve: BM25 top-20 → CrossEncoder → top-5
    retrieved = retrieve_with_reranking(q)

    # Generate
    answer    = generate_answer(q, retrieved)
    predicted = extract_label(answer)

    demo_results.append({
        'idx': i, 'question': q, 'ground_truth': gt,
        'predicted_label': predicted, 'is_correct': predicted == gt,
        'answer': answer,
        'bm25_scores'    : [r.score for r in retrieved],
        'reranker_scores': [r.reranker_score for r in retrieved],
    })

    status = 'BENAR' if predicted == gt else 'SALAH'
    print(f'\n[{i}] {q[:70]}')
    print(f'     GT={gt} | Pred={predicted} | {status}')
    print(f'     Reranker scores: {[f"{r.reranker_score:.3f}" for r in retrieved]}')
    print(f'     Jawaban: {answer[:100]}...')

demo_acc = sum(r['is_correct'] for r in demo_results) / DEMO_SIZE
print(f'\nDemo accuracy: {demo_acc:.0%} ({sum(r["is_correct"] for r in demo_results)}/{DEMO_SIZE})')
print('Pipeline CR siap untuk Fase 1.')

DEMO: 5 sampel pertama (RAG + Context Reranking)

[0] Do mitochondria play a role in remodelling lace plant leaves during pr
     GT=yes | Pred=yes | BENAR
     Reranker scores: ['6.258', '-5.053', '-9.369', '-9.751', '-10.405']
     Jawaban: Based on the provided scientific abstracts, it appears that mitochondria play a role in programmed c...

[1] Landolt C and snellen e acuity: differences in strabismus amblyopia?
     GT=no | Pred=yes | SALAH
     Reranker scores: ['8.947', '5.589', '2.035', '-3.391', '-10.751']
     Jawaban: The study abstracts suggest that there are small differences between Landolt C acuity (LR) and Snell...

[2] Syncope during bathing in infants, a pediatric form of water-induced u
     GT=yes | Pred=yes | BENAR
     Reranker scores: ['0.859', '-7.613', '-7.983', '-9.676', '-10.308']
     Jawaban: The question of syncope during bathing in infants, a pediatric form of water-induced urticaria, can ...

[3] Are the long-term results of the transanal pull-through e

---
## Fase 1: Generate Semua Jawaban (500 Sampel)

BM25 top-20 → CrossEncoder rerank → top-5 → LLM generate.
Disimpan inkremental setiap 10 sampel — resume-able jika terputus.

Field tambahan vs baseline: `reranker_scores`, `bm25_scores_candidates` (skor BM25 top-20 sebelum rerank)

> Estimasi: ~4–6 jam (CrossEncoder inference di CPU lebih lambat dari BM25 saja)

In [12]:
if PHASE1_PATH.exists():
    with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
        phase1_results = json.load(f)['results']
    start_from = len(phase1_results)
    print(f'Resume Fase 1: {start_from}/{MAX_SAMPLES} sudah selesai.')
else:
    phase1_results, start_from = [], 0
    print(f'Memulai Fase 1: {MAX_SAMPLES} sampel (BM25 top-{TOP_K_CANDIDATES} → CR → top-{TOP_K_RETRIEVAL}).')

if start_from < MAX_SAMPLES:
    print(f'Memproses {MAX_SAMPLES - start_from} sampel tersisa...\n')
    t_start = time.time()
    for i in range(start_from, MAX_SAMPLES):
        s          = pubmedqa_data[i]
        q, gt, ref = s['question'], s['final_decision'], s['long_answer']

        retrieved  = retrieve_with_reranking(q)
        answer     = generate_answer(q, retrieved)
        predicted  = extract_label(answer)

        phase1_results.append({
            'idx'             : i,
            'pubid'           : str(s['pubid']),
            'question'        : q,
            'ground_truth'    : gt,
            'predicted_label' : predicted,
            'is_correct'      : predicted == gt,
            'answer'          : answer,
            'contexts'        : [r.document.text for r in retrieved],
            'reference'       : ref,
            'retrieval_scores': [r.score for r in retrieved],
            'reranker_scores' : [r.reranker_score for r in retrieved],
        })

        if (i + 1) % 10 == 0 or i == MAX_SAMPLES - 1:
            with open(PHASE1_PATH, 'w', encoding='utf-8') as f:
                json.dump({'config': CONFIG_NAME,
                           'timestamp': datetime.now().isoformat(),
                           'max_samples': MAX_SAMPLES, 'completed': i+1,
                           'results': phase1_results}, f, indent=2, ensure_ascii=False)
            done = i + 1
            acc  = sum(r['is_correct'] for r in phase1_results) / done
            eta  = (time.time()-t_start) / done * (MAX_SAMPLES-done) / 60
            print(f'  [{done:3d}/{MAX_SAMPLES}] Akurasi: {acc:.1%} | pred={predicted}, gt={gt} | ETA {eta:.1f} mnt')
    print(f'\nFase 1 selesai! Disimpan ke {PHASE1_PATH}')
else:
    print(f'Fase 1 sudah selesai ({MAX_SAMPLES} sampel).')

Memulai Fase 1: 500 sampel (BM25 top-20 → CR → top-5).
Memproses 500 sampel tersisa...

  [ 10/500] Akurasi: 40.0% | pred=yes, gt=yes | ETA 435.7 mnt
  [ 20/500] Akurasi: 60.0% | pred=yes, gt=yes | ETA 407.6 mnt
  [ 30/500] Akurasi: 66.7% | pred=yes, gt=yes | ETA 397.0 mnt
  [ 40/500] Akurasi: 57.5% | pred=yes, gt=no | ETA 378.1 mnt
  [ 50/500] Akurasi: 58.0% | pred=yes, gt=no | ETA 372.4 mnt
  [ 60/500] Akurasi: 55.0% | pred=yes, gt=yes | ETA 358.4 mnt
  [ 70/500] Akurasi: 55.7% | pred=yes, gt=yes | ETA 350.8 mnt
  [ 80/500] Akurasi: 57.5% | pred=yes, gt=yes | ETA 347.4 mnt
  [ 90/500] Akurasi: 58.9% | pred=yes, gt=maybe | ETA 342.2 mnt
  [100/500] Akurasi: 61.0% | pred=yes, gt=yes | ETA 341.9 mnt
  [110/500] Akurasi: 59.1% | pred=no, gt=yes | ETA 332.1 mnt
  [120/500] Akurasi: 55.8% | pred=no, gt=yes | ETA 322.9 mnt
  [130/500] Akurasi: 53.8% | pred=yes, gt=maybe | ETA 314.0 mnt
  [140/500] Akurasi: 55.0% | pred=yes, gt=yes | ETA 307.7 mnt
  [150/500] Akurasi: 54.0% | pred=yes, gt=no

### Analisis Fase 1 (Label Accuracy)

In [13]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']
n         = len(results_p1)
n_correct = sum(r['is_correct'] for r in results_p1)
gts       = [r['ground_truth']    for r in results_p1]
preds     = [r['predicted_label'] for r in results_p1]

print(f'ANALISIS FASE 1 -- {n} sampel (RAG + Context Reranking)')
print('=' * 55)
print(f'Label Accuracy    : {n_correct}/{n} = {n_correct/n:.1%}')
print(f'Hallucination Rate: {(n-n_correct)/n:.1%}\n')
print(f'  {"Label":<8} | {"Ground Truth":>12} | {"Prediksi":>12}')
print(f'  {"-"*8}-+{"-"*14}-+{"-"*12}')
for lbl in ['yes','no','maybe']:
    g, p = gts.count(lbl), preds.count(lbl)
    print(f'  {lbl:<8} | {g:>10} ({g/n:.0%}) | {p:>10} ({p/n:.0%})')

print('\nConfusion Matrix (baris=GT, kolom=Prediksi):')
lbls = ['yes','no','maybe']
print('  ' + f'{"GT/Pred":>8}' + ''.join(f'{l:>8}' for l in lbls))
for gt_l in lbls:
    row = f'  {gt_l:>8}'
    for pr_l in lbls:
        cnt = sum(1 for r in results_p1 if r['ground_truth']==gt_l and r['predicted_label']==pr_l)
        row += f'{cnt:>8}'
    print(row)

# Statistik reranker score
all_rr_scores = [s for r in results_p1 for s in r.get('reranker_scores', [])]
all_bm25_scores = [s for r in results_p1 for s in r.get('retrieval_scores', [])]
if all_rr_scores:
    print(f'\nStatistik Reranker Score (top-{TOP_K_RETRIEVAL} per sampel):')
    print(f'  Mean  : {np.mean(all_rr_scores):.4f}')
    print(f'  Median: {np.median(all_rr_scores):.4f}')
    print(f'  Min   : {np.min(all_rr_scores):.4f}')
    print(f'  Max   : {np.max(all_rr_scores):.4f}')

ANALISIS FASE 1 -- 500 sampel (RAG + Context Reranking)
Label Accuracy    : 270/500 = 54.0%
Hallucination Rate: 46.0%

  Label    | Ground Truth |     Prediksi
  ---------+---------------+------------
  yes      |        275 (55%) |        432 (86%)
  no       |        159 (32%) |         50 (10%)
  maybe    |         66 (13%) |         18 (4%)

Confusion Matrix (baris=GT, kolom=Prediksi):
   GT/Pred     yes      no   maybe
       yes     245      23       7
        no     127      23       9
     maybe      60       4       2

Statistik Reranker Score (top-5 per sampel):
  Mean  : -2.4110
  Median: -3.5116
  Min   : -11.3816
  Max   : 10.8021


---
## Fase 2: Evaluasi RAGAS (Referensi)

Cell ini menggunakan RAGAS library langsung — kemungkinan ada NaN.
Gunakan hanya sebagai referensi. **Yang dipakai untuk analisis utama: Fase 2 Custom (cell berikutnya).**

> Estimasi: ~2–3 jam untuk 100 sampel

In [ ]:
MAX_RAGAS_SAMPLES = 100
PHASE2_RAGAS_PATH = RESULTS_DIR / f'{CONFIG_NAME}_phase2_ragas.json'

if PHASE2_RAGAS_PATH.exists():
    with open(PHASE2_RAGAS_PATH, 'r', encoding='utf-8') as f:
        phase2_ragas = json.load(f)['results']
    done_ragas = {r['idx'] for r in phase2_ragas}
    print(f'Resume RAGAS: {len(done_ragas)}/{MAX_RAGAS_SAMPLES} selesai.')
else:
    phase2_ragas, done_ragas = [], set()
    print(f'Mulai evaluasi RAGAS: {MAX_RAGAS_SAMPLES} sampel.')

with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    p1_ragas = json.load(f)['results'][:MAX_RAGAS_SAMPLES]

remaining_ragas = [r for r in p1_ragas if r['idx'] not in done_ragas]
print(f'Sisa: {len(remaining_ragas)} sampel\n')

t0 = time.time()
for i, r in enumerate(remaining_ragas):
    scores = evaluate_ragas_fast(r['question'], r['answer'], r['contexts'], r['reference'])
    phase2_ragas.append({
        'idx': r['idx'], 'ground_truth': r['ground_truth'],
        'predicted_label': r['predicted_label'], 'is_correct': r['is_correct'],
        **scores
    })
    if (i + 1) % 5 == 0 or i == len(remaining_ragas) - 1:
        with open(PHASE2_RAGAS_PATH, 'w', encoding='utf-8') as f:
            json.dump({'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                       'results': phase2_ragas}, f, indent=2, ensure_ascii=False)
        nan_f = sum(1 for x in phase2_ragas if x.get('faithfulness') != x.get('faithfulness'))
        print(f'  [{i+1:3d}/{len(remaining_ragas)}] idx={r["idx"]} | NaN_faith={nan_f}')

print(f'\nRAGAS selesai! -> {PHASE2_RAGAS_PATH}')

---
## Fase 2 Custom (Zero-NaN) — 500 Sampel

Gunakan `evaluate_custom` — dijamin tidak NaN. Resume-able.
Simpan ke `cr_phase2_custom.json`.

> Estimasi: ~3–5 jam untuk 500 sampel

In [ ]:
MAX_CUSTOM_SAMPLES  = 500
PHASE2_CUSTOM_PATH  = RESULTS_DIR / f'{CONFIG_NAME}_phase2_custom.json'

with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    p1_custom = json.load(f)['results'][:MAX_CUSTOM_SAMPLES]

if PHASE2_CUSTOM_PATH.exists():
    with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
        p2_custom = json.load(f)['results']
    done_custom = {r['idx'] for r in p2_custom}
    print(f'Resume: {len(done_custom)}/{MAX_CUSTOM_SAMPLES} selesai.')
else:
    p2_custom, done_custom = [], set()
    print(f'Mulai: {MAX_CUSTOM_SAMPLES} sampel (custom zero-NaN).')

remaining = [r for r in p1_custom if r['idx'] not in done_custom]
print(f'Sisa: {len(remaining)} | Estimasi ~{len(remaining)*0.5/60:.1f} jam\n')

t0 = time.time()
for i, r in enumerate(remaining):
    scores = evaluate_custom(r['question'], r['answer'], r['contexts'], r['reference'])
    p2_custom.append({
        'idx': r['idx'], 'ground_truth': r['ground_truth'],
        'predicted_label': r['predicted_label'], 'is_correct': r['is_correct'],
        **scores
    })
    if (i + 1) % 5 == 0 or i == len(remaining) - 1:
        with open(PHASE2_CUSTOM_PATH, 'w', encoding='utf-8') as f:
            json.dump({
                'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                'max_samples': MAX_CUSTOM_SAMPLES,
                'metrics': ['faithfulness', 'context_recall'],
                'evaluator': 'custom_zero_nan',
                'results': p2_custom
            }, f, indent=2, ensure_ascii=False)
        done  = i + 1
        total = len(remaining)
        eta   = (time.time()-t0)/done*(total-done)/60 if done < total else 0
        avg_f = sum(x['faithfulness']   for x in p2_custom) / len(p2_custom)
        avg_r = sum(x['context_recall'] for x in p2_custom) / len(p2_custom)
        print(f'  [{done:3d}/{total}] idx={r["idx"]} | '
              f'faith={scores["faithfulness"]:.3f} | cr={scores["context_recall"]:.3f} | '
              f'avg_f={avg_f:.3f} | avg_cr={avg_r:.3f} | ETA {eta:.1f} mnt')

print(f'\nSelesai! -> {PHASE2_CUSTOM_PATH}')

n   = len(p2_custom)
acc = sum(r['is_correct'] for r in p2_custom) / n
avg_f  = sum(r['faithfulness']   for r in p2_custom) / n
avg_cr = sum(r['context_recall'] for r in p2_custom) / n
nan_f  = sum(1 for r in p2_custom if r['faithfulness']   != r['faithfulness'])
nan_cr = sum(1 for r in p2_custom if r['context_recall'] != r['context_recall'])
print(f'\nRingkasan ({n} sampel, zero-NaN evaluator):')
print(f'  Label Accuracy : {acc:.1%}')
print(f'  Faithfulness   : {avg_f:.4f}  NaN={nan_f}')
print(f'  Context Recall : {avg_cr:.4f} NaN={nan_cr}')
print(f'\nBaris tabel skripsi:')
print(f'  | {CONFIG_NAME.upper()} | {acc:.3f} | {avg_f:.3f} | {avg_cr:.3f} |')

Mulai: 500 sampel (custom zero-NaN).
Sisa: 500 | Estimasi ~4.2 jam



---
## Ringkasan Metrik Evaluasi Context Reranking

In [ ]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']
with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
    results_p2 = json.load(f)['results']
p2_lu = {r['idx']: r for r in results_p2}

df = pd.DataFrame([{
    'idx'             : r['idx'],
    'ground_truth'    : r['ground_truth'],
    'predicted_label' : r['predicted_label'],
    'is_correct'      : r['is_correct'],
    'faithfulness'    : p2_lu.get(r['idx'], {}).get('faithfulness',   float('nan')),
    'context_recall'  : p2_lu.get(r['idx'], {}).get('context_recall', float('nan')),
    'avg_reranker_score': float(np.mean(r.get('reranker_scores', [0]))),
    'avg_bm25_score'    : float(np.mean(r.get('retrieval_scores', [0]))),
} for r in results_p1])

n, n_correct = len(df), int(df['is_correct'].sum())
label_acc  = n_correct / n
hallu_rate = 1 - label_acc
avg_f  = df['faithfulness'].mean()
avg_cr = df['context_recall'].mean()

print('=' * 60)
print(f'  CONTEXT RERANKING (BM25 top-{TOP_K_CANDIDATES} → CR top-{TOP_K_RETRIEVAL}) — {n} sampel')
print('=' * 60)
print(f'  {"Metrik":<32} {"Nilai":>10}')
print(f'  {"-"*42}')
print(f'  {"Label Accuracy":<32} {label_acc:>9.1%}')
print(f'  {"Hallucination Rate":<32} {hallu_rate:>9.1%}')
print(f'  {"-"*42}')
print(f'  {"Faithfulness (custom)":<32} {avg_f:>10.4f}')
print(f'  {"Context Recall (custom)":<32} {avg_cr:>10.4f}')
print(f'  {"Avg Reranker Score":<32} {df["avg_reranker_score"].mean():>10.4f}')
print(f'  {"Avg BM25 Score (top-5)":<32} {df["avg_bm25_score"].mean():>10.4f}')
print('=' * 60)
print('\nPer-label accuracy:')
for lbl in ['yes','no','maybe']:
    sub = df[df['ground_truth']==lbl]
    if len(sub):
        print(f'  {lbl:>5}: {sub["is_correct"].mean():.1%} ({int(sub["is_correct"].sum())}/{len(sub)}) '
              f'| faithfulness={sub["faithfulness"].mean():.3f} '
              f'| context_recall={sub["context_recall"].mean():.3f}')

---
## Perbandingan: Baseline vs QR vs Context Reranking

Jalankan setelah semua tiga notebook selesai.

In [ ]:
configs_available = {
    'Baseline (BM25)'       : (BASELINE_PHASE1_PATH, BASELINE_PHASE2_CUSTOM_PATH),
    'BM25 + QR'             : (QR_PHASE1_PATH,       QR_PHASE2_CUSTOM_PATH),
    f'BM25 + CR (top-{TOP_K_CANDIDATES}→{TOP_K_RETRIEVAL})': (PHASE1_PATH, PHASE2_CUSTOM_PATH),
}

dfs = {}
for cfg_name, (p1_path, p2_path) in configs_available.items():
    if not p1_path.exists() or not p2_path.exists():
        print(f'[SKIP] {cfg_name}: file belum tersedia.')
        continue
    with open(p1_path) as f:
        p1 = json.load(f)['results']
    with open(p2_path) as f:
        p2 = json.load(f)['results']
    p2_lu = {r['idx']: r for r in p2}
    dfs[cfg_name] = pd.DataFrame([{
        'idx'            : r['idx'],
        'ground_truth'   : r['ground_truth'],
        'predicted_label': r['predicted_label'],
        'is_correct'     : r['is_correct'],
        'faithfulness'   : p2_lu.get(r['idx'], {}).get('faithfulness',   float('nan')),
        'context_recall' : p2_lu.get(r['idx'], {}).get('context_recall', float('nan')),
    } for r in p1])

if not dfs:
    print('Belum ada data untuk dibandingkan.')
else:
    print('=' * 80)
    print('PERBANDINGAN: BASELINE vs QR vs CONTEXT RERANKING')
    print('=' * 80)
    hdr = f'  {"Konfigurasi":<30} | {"Acc":>7} | {"Hallu":>6} | {"Faith":>7} | {"Ctx.R":>7}'
    print(hdr)
    print('  ' + '-' * 66)
    for cfg_name, dfc in dfs.items():
        acc  = dfc['is_correct'].mean()
        hr   = 1 - acc
        f_   = dfc['faithfulness'].mean()
        cr   = dfc['context_recall'].mean()
        print(f'  {cfg_name:<30} | {acc:>6.1%} | {hr:>5.1%} | {f_:>7.4f} | {cr:>7.4f}')
    print('=' * 80)

    if 'Baseline (BM25)' in dfs and f'BM25 + CR (top-{TOP_K_CANDIDATES}→{TOP_K_RETRIEVAL})' in dfs:
        bl  = dfs['Baseline (BM25)']
        cr_ = dfs[f'BM25 + CR (top-{TOP_K_CANDIDATES}→{TOP_K_RETRIEVAL})']
        print('\nDelta CR vs Baseline:')
        for metrik, bl_val, cr_val in [
            ('Label Accuracy',   bl['is_correct'].mean(),     cr_['is_correct'].mean()),
            ('Faithfulness',     bl['faithfulness'].mean(),   cr_['faithfulness'].mean()),
            ('Context Recall',   bl['context_recall'].mean(), cr_['context_recall'].mean()),
        ]:
            delta = cr_val - bl_val
            label = 'LEBIH BAIK ↑' if delta > 0.001 else ('LEBIH BURUK ↓' if delta < -0.001 else 'SAMA')
            print(f'  {metrik:<22}: {delta:+.4f}  ({label})')

    print('\nPer-label accuracy:')
    print(f'  {"Label":<7}' + ''.join(f' | {n:>28}' for n in dfs.keys()))
    print('  ' + '-' * (7 + 31 * len(dfs)))
    for lbl in ['yes', 'no', 'maybe']:
        row = f'  {lbl:<7}'
        for dfc in dfs.values():
            sub = dfc[dfc['ground_truth'] == lbl]
            if len(sub):
                row += f' | {sub["is_correct"].mean():>6.1%} (n={len(sub):>3})            '
            else:
                row += f' | {"N/A":>28}'
        print(row)

---
## Analisis Kualitas Reranking

Inspeksi kualitatif: seberapa besar CrossEncoder mengubah urutan dari BM25?

In [ ]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    cr_p1 = json.load(f)['results']

# Bandingkan: apakah reranker mengubah urutan?
# Simpan raw BM25 top-20 candidates untuk analisis
# (field 'retrieval_scores' di fase 1 adalah scores SETELAH rerank)
# Ukur reranker score spread: perbedaan antara dok terbaik dan terburuk dalam top-5

rr_spreads = []
for r in cr_p1:
    rr = r.get('reranker_scores', [])
    if len(rr) >= 2:
        rr_spreads.append(max(rr) - min(rr))

print('Analisis Reranker Score (top-5 per sampel):')
print('=' * 50)
all_rr = [s for r in cr_p1 for s in r.get('reranker_scores', [])]
print(f'  Mean score  : {np.mean(all_rr):.4f}')
print(f'  Std score   : {np.std(all_rr):.4f}')
print(f'  Min score   : {np.min(all_rr):.4f}')
print(f'  Max score   : {np.max(all_rr):.4f}')
if rr_spreads:
    print(f'  Avg spread (max-min per sampel): {np.mean(rr_spreads):.4f}')

print('\nContoh 10 sampel (reranker scores top-5):')
print('=' * 70)
for r in cr_p1[:10]:
    rr_scores = r.get('reranker_scores', [])
    bm_scores = r.get('retrieval_scores', [])
    status    = 'BENAR' if r['is_correct'] else 'SALAH'
    print(f'[{r["idx"]:3d}] GT={r["ground_truth"]} | Pred={r["predicted_label"]} | {status}')
    print(f'       Q: {r["question"][:70]}')
    if rr_scores:
        print(f'       Reranker: {[f"{s:.3f}" for s in rr_scores]}')
    if bm_scores:
        print(f'       BM25   : {[f"{s:.1f}" for s in bm_scores]}')
    print()